In [87]:
import os
import numpy as np
import mne
from pathlib import Path

In [88]:
subjects_dir = mne.datasets.fetch_fsaverage(verbose=True)
subjects_dir = Path(subjects_dir).parent
subjects_dir
subject = 'fsaverage'

0 files missing from root.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data
0 files missing from bem.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage


In [89]:
src = mne.setup_source_space(subject = subject, spacing='ico5', subjects_dir=subjects_dir, add_dist=False)

Setting up the source space with the following parameters:

SUBJECTS_DIR = C:\Users\hugop\mne_data\MNE-fsaverage-data
Subject      = fsaverage
Surface      = white
Icosahedron subdivision grade 5

>>> 1. Creating the source space...

Doing the icosahedral vertex picking...
Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.white...
Mapping lh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.sphere...
Setting up the triangulation for the decimated surface...
loaded lh.white 10242/163842 selected to source space (ico = 5)

Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.white...
Mapping rh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.sphere...
Setting up the triangulation for the decimated surface...
loaded rh.white 10242/163842 selected to 

In [90]:
hemi = 0 # 0 = left | 1 = right

rr = src[hemi]['rr']
vertno = src[hemi]['vertno'] 
coords = rr[vertno] 

In [91]:
y_ant = 0.02    # avant
y_post = -0.04  # arrière
z_sup = 0.02    # haut
z_inf = -0.02   # bas

x = coords[:, 0]
y = coords[:, 1]
z = coords[:, 2]

In [92]:
print(coords.min(axis=0))
print(coords.max(axis=0))

xmin, ymin, zmin = coords.min(axis=0)
xmax, ymax, zmax = coords.max(axis=0)

temporal_mask = (z < z_inf) & (y > y_post) & (y < y_ant)
temporal_vertices = vertno[temporal_mask]

occipital_mask = y < y_post
occipital_vertices = vertno[occipital_mask]

frontal_mask = y > y_ant
frontal_vertices = vertno[frontal_mask]

used = temporal_mask | occipital_mask | frontal_mask
parietal_mask = ~used
parietal_vertices = vertno[parietal_mask]

[-0.0656627  -0.10240875 -0.04242961]
[0.00179359 0.0647402  0.07524361]


In [93]:
temp_coords = coords[temporal_mask]
temp_vert = vertno[temporal_mask]

temp_ant = temp_vert[temp_coords[:, 1] > 0]
temp_post = temp_vert[temp_coords[:, 1] <= 0]

temp_sup = temp_vert[temp_coords[:, 2] > -0.04]
temp_inf = temp_vert[temp_coords[:, 2] <= -0.04]


In [94]:
atlas = {
    "frontal": frontal_vertices,
    "parietal": parietal_vertices,
    "occipital": occipital_vertices,
    "temporal": temporal_vertices,
    
    "temporal_ant": temp_ant,
    "temporal_post": temp_post,
    "temporal_sup": temp_sup,
    "temporal_inf": temp_inf,
}

In [95]:

label = mne.Label(
    vertices=temporal_vertices,
    hemi='lh',
    name='temporal_custom'
)


In [96]:
def make_mask(coords, px = (0,1),py= (0,1),pz= (0,1)):
    
    def calcul(min,max,p):
        return min + p * (max - min)
    
    minx,miny,minz = coords.min(axis=0)
    maxx, maxy, maxz = coords.max(axis=0)

    x = coords[:, 0]
    y = coords[:, 1]
    z = coords[:, 2]

    x_inf = calcul(minx,maxx,px[0])
    x_sup = calcul(minx,maxx,px[1])

    y_inf = calcul(miny,maxy,py[0])
    y_sup = calcul(miny,maxy,py[1])

    z_inf = calcul(minz,maxz,pz[0])
    z_sup = calcul(minz,maxz,pz[1])


    mask = ((x <= x_sup) & (x >= x_inf) & 
             (y <= y_sup) & (y >= y_inf) & 
             (z <= z_sup) & (z >= z_inf))

    return mask

def fusion_mask_list(liste):
    
    def fusion_mask(mask_1, mask_2) :
        mask_out = []
        if len(mask_1) == 0 or mask_1 is None:
            return mask_2
        else :
            for m1, m2 in zip(mask_1,mask_2):
                if m1 == False and m2 == False:
                    mask_out.append(False)
                else :
                    mask_out.append(True)
        return mask_out
    
    mask_out = []
    for mask in liste:
        mask_out = fusion_mask(mask_out, mask)
        
    return np.array(mask_out)


In [386]:
brain = mne.viz.Brain('fsaverage', hemi='lh', surf='inflated')
###################### coordonee temporal ############################
mask_temp_2 = make_mask(coords,py=(0.35,0.47),pz=(0,0.5),px = (0,0.37))
mask_temp_1_bis = make_mask(coords,py=(0.47,0.6),pz=(0,0.42),px = (0,0.36))
mask_temp_1 = make_mask(coords,py=(0.47,0.75),pz=(0,0.366),px = (0,0.36))
mask_temp_3 = make_mask(coords,py=(0.22,0.35),pz=(0,0.55),px = (0,0.45))

###################### coordonee occipital ############################

mask_occ_1 = make_mask(coords,py=(0,0.13),pz=(0,0.7),px = (0.58,1))
mask_occ_2 = make_mask(coords,py=(0,0.22),pz=(0,0.7),px = (0,0.58))
mask_occ_3 = make_mask(coords,py=(0,0.18),pz=(0.65,0.8),px = (0,0.9))
mask_occ_4 = make_mask(coords,py=(0,0.2),pz=(0.45,1),px = (0,0.65))
mask_occ_5 = make_mask(coords,py=(0.2,0.26),pz=(0.58,0.715),px = (0.55,0.69))

###################### coordonee TPJ - Parietal ############################

mask_TPJ_parietal1 = make_mask(coords,py=(0,0.22),pz=(0.7,1),px = (0,0.7))
mask_TPJ_parietal2 = make_mask(coords,py=(0.22,0.42),pz=(0.55,1),px = (0,0.7))
mask_TPJ_parietal3 = make_mask(coords,py=(0,0.22),pz=(0.7,1),px = (0,0.87))
mask_TPJ_parietal4 = make_mask(coords,py=(0.22,0.32),pz=(0.83,1),px = (0,0.85))
mask_TPJ_parietal5 = make_mask(coords,py=(0.32,0.43),pz=(0.89,1),px = (0,0.95))

###################### coordonee globales ############################

mask_TPJ_parietal = fusion_mask_list([mask_TPJ_parietal1,mask_TPJ_parietal2,mask_TPJ_parietal3,mask_TPJ_parietal4])
mask_temp = fusion_mask_list([mask_temp_1,mask_temp_1_bis,mask_temp_2,mask_temp_3])
mask_occ = fusion_mask_list([mask_occ_1,mask_occ_2,mask_occ_3,mask_occ_4,mask_occ_5])


temp_1 = mne.Label(vertno[mask_temp_1], hemi='lh', name='test')
temp_1_bis = mne.Label(vertno[mask_temp_1_bis], hemi='lh', name='test')
temp_2 = mne.Label(vertno[mask_temp_2], hemi='lh', name='test')
temp_3 = mne.Label(vertno[mask_temp_3], hemi='lh', name='test')

test_tpj_1 = mne.Label(vertno[mask_TPJ_parietal1], hemi='lh', name='test2')
test_tpj_2 = mne.Label(vertno[mask_TPJ_parietal2], hemi='lh', name='test2')
test_tpj_3 = mne.Label(vertno[mask_TPJ_parietal3], hemi='lh', name='test2')
test_tpj_4 = mne.Label(vertno[mask_TPJ_parietal4], hemi='lh', name='test2')
test_tpj_5 = mne.Label(vertno[mask_TPJ_parietal5], hemi='lh', name='test2')

occ_1 = mne.Label(vertno[mask_occ_1], hemi='lh', name='test2')
occ_2 = mne.Label(vertno[mask_occ_2], hemi='lh', name='test2')
occ_3 = mne.Label(vertno[mask_occ_3], hemi='lh', name='test2')
occ_4 = mne.Label(vertno[mask_occ_4], hemi='lh', name='test2')
occ_5 = mne.Label(vertno[mask_occ_5], hemi='lh', name='test2')



################ affichage sous temporal ###################

# brain.add_label(temp_1, color='yellow')
# brain.add_label(temp_1_bis, color='green')
# brain.add_label(temp_2, color='red')
# brain.add_label(temp_3, color='blue')

################ affichage sous TPJ ###################

# brain.add_label(test_tpj_1, color='yellow')
# brain.add_label(test_tpj_2, color='green')
# brain.add_label(test_tpj_3, color='yellow')
# brain.add_label(test_tpj_4, color='blue')
# brain.add_label(test_tpj_5, color='purple')

################ affichage sous occipital ###################

# brain.add_label(occ_1, color='red')
# brain.add_label(occ_2, color='blue')
# brain.add_label(occ_3, color='yellow')
# brain.add_label(occ_4, color='green')
# brain.add_label(occ_5, color='purple')

################## affichage globale ##########################

TPJ_parietal = mne.Label(vertno[mask_TPJ_parietal], hemi='lh', name='test2')
occ = mne.Label(vertno[mask_occ], hemi='lh', name='test2')
temp =  mne.Label(vertno[mask_temp], hemi='lh', name='test')

brain.add_label(TPJ_parietal, color='yellow')
brain.add_label(temp, color='blue')
brain.add_label(occ, color='red')


